# Descenso de Gradiente — Prototipo

Proyecto: Optimización de algoritmos de Inteligencia Artificial mediante Cálculo Diferencial.

Este notebook implementa el algoritmo de Descenso de Gradiente desde cero (NumPy) y lo compara contra una búsqueda aleatoria de parámetros.

## 1. Generación del dataset

Dataset sintético de clasificación binaria, exportado a `dataset.csv`.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification

RANDOM_STATE = 42

X, y = make_classification(
    n_samples=500,
    n_features=2,
    n_informative=2,
    n_redundant=0,
    n_clusters_per_class=1,
    class_sep=1.5,
    random_state=RANDOM_STATE,
)

dataset = pd.DataFrame(X, columns=["x1", "x2"])
dataset["y"] = y
dataset.to_csv("dataset.csv", index=False)

print(f"Dataset generado: {dataset.shape[0]} muestras, {dataset.shape[1] - 1} features")
dataset.head()


## 2. Función de costo (MSE) y su derivada

Implementación explícita del Error Cuadrático Medio y de ∂J/∂w.

In [ ]:
def sigmoid(z):
    """sigma(z) = 1 / (1 + e^-z), mapea la salida lineal a un rango [0, 1]."""
    return 1 / (1 + np.exp(-z))


def predict(X, W, b):
    """y_hat = sigma(X.W + b)"""
    z = X @ W + b
    return sigmoid(z)


def cost_function(X, y, W, b):
    """J(W, b) = (1/n) * sum((y_hat - y)^2)  -- Error Cuadratico Medio."""
    y_hat = predict(X, W, b)
    return np.mean((y_hat - y) ** 2)


def gradient(X, y, W, b):
    """
    Derivada explicita de J respecto a W y b (regla de la cadena):
        dJ/dy_hat = (2/n) * (y_hat - y)
        dy_hat/dz = y_hat * (1 - y_hat)          <- derivada de la sigmoide
        dz/dW = X ; dz/db = 1

        dJ/dW = (2/n) * sum((y_hat - y) * y_hat*(1-y_hat) * x)
        dJ/db = (2/n) * sum((y_hat - y) * y_hat*(1-y_hat))
    """
    n = X.shape[0]
    y_hat = predict(X, W, b)
    delta = (2 / n) * (y_hat - y) * y_hat * (1 - y_hat)
    dW = X.T @ delta
    db = np.sum(delta)
    return dW, db


# Prueba rapida con pesos iniciales en cero
W_test = np.zeros(X.shape[1])
b_test = 0.0
print(f"Costo inicial (W=0, b=0): {cost_function(X, y, W_test, b_test):.4f}")
dW_test, db_test = gradient(X, y, W_test, b_test)
print(f"Gradiente inicial -> dW: {dW_test}, db: {db_test:.4f}")


## 3. Descenso de Gradiente

Bucle iterativo W ← W − α·∂J/∂w, registrando el error en cada iteración.

In [ ]:
ALPHA = 1.0        # tasa de aprendizaje
MAX_ITER = 5000     # tope de iteraciones
TOL = 1e-6          # umbral de convergencia: |J(t) - J(t-1)| < TOL


def gradient_descent(X, y, alpha=ALPHA, max_iter=MAX_ITER, tol=TOL):
    """
    Descenso de Gradiente desde cero:
        W(t+1) = W(t) - alpha * dJ/dW
        b(t+1) = b(t) - alpha * dJ/db
    Se detiene cuando el costo deja de mejorar mas que `tol`, o al llegar a max_iter.
    Devuelve W, b finales y el historial de costo (uno por iteracion).
    """
    W = np.zeros(X.shape[1])
    b = 0.0
    cost_history = [cost_function(X, y, W, b)]

    for i in range(1, max_iter + 1):
        dW, db = gradient(X, y, W, b)
        W = W - alpha * dW
        b = b - alpha * db
        cost_history.append(cost_function(X, y, W, b))

        if abs(cost_history[-2] - cost_history[-1]) < tol:
            break

    return W, b, cost_history


W_gd, b_gd, cost_history_gd = gradient_descent(X, y)
iters_gd = len(cost_history_gd) - 1

y_pred_gd = (predict(X, W_gd, b_gd) >= 0.5).astype(int)
accuracy_gd = np.mean(y_pred_gd == y)

print(f"Descenso de Gradiente -> iteraciones: {iters_gd}")
print(f"Costo inicial: {cost_history_gd[0]:.4f} | Costo final: {cost_history_gd[-1]:.4f}")
print(f"W final: {W_gd}, b final: {b_gd:.4f}")
print(f"Precision final: {accuracy_gd:.4f}")


## 4. Búsqueda Aleatoria (baseline)

Mismo problema resuelto probando parámetros al azar, para comparación.

In [ ]:
# TODO: implementar búsqueda aleatoria


## 5. Métricas

CPU, RAM y tiempo real (medidos con `psutil`), iteraciones hasta converger, precisión final — para ambos métodos.

In [ ]:
# TODO: medir e imprimir métricas comparativas


## 6. Gráficos

Curva de error vs. iteración y comparación entre métodos, exportados como PNG para la diapositiva 12 del PPT.

In [ ]:
# TODO: graficar y exportar resultados
